1. Notebook setup - drive, HF login, paths, seed

In [4]:
!pip install -q datasets huggingface_hub datasketch openpyxl

from google.colab import drive, userdata
from huggingface_hub import login
import os

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE        = "/content/drive/MyDrive/Research-WSD-New"
OUT         = f"{BASE}/extraction_v3"
os.makedirs(OUT, exist_ok=True)

RANDOM_SEED = 42
RESERVOIR_K = 1000   # sentences kept per word per source during the scan

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load target words

In [6]:
import re, itertools, unicodedata
from collections import defaultdict
import openpyxl

WORDS_XLSX = "/content/drive/MyDrive/Research-WSD-New/sinhala_polysemy_words.xlsx"
REMOVED    = {"වේ"}

def load_target_words(path=WORDS_XLSX, sheet="words"):
    """Reads No | Word | forms. Returns the word list (in sheet order), {word: No},
    {word: [spellings]} and {word: [forms]}. 'පස / පස්' stays ONE word."""
    ws = openpyxl.load_workbook(path)[sheet]
    rows = ws.iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]          # 'No ' has a trailing space
    i_no, i_word = header.index("No"), header.index("Word")
    # the forms column is called "forms (rule-based)" in the sheet; "Word forms" also accepted
    i_forms = next(i for i, h in enumerate(header) if h in ("Word forms", "forms (rule-based)"))
    words, numbers, spellings, forms = [], {}, {}, {}
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w = str(r[i_word]).strip()
        sp = [s.strip() for s in re.split(r"[/,]", w) if s.strip()]    # "පස / පස්" -> ["පස", "පස්"]
        if w in REMOVED or any(s in REMOVED for s in sp):
            continue
        if w in numbers:
            raise ValueError(f"Duplicate word in sheet: {w}")
        f = [x.strip() for x in str(r[i_forms] or "").split(",") if x.strip()]
        if not f:
            raise ValueError(f"No forms for {w}")
        words.append(w)
        numbers[w] = int(float(r[i_no]))
        spellings[w] = sp
        forms[w] = list(dict.fromkeys(sp + f))
    return words, numbers, spellings, forms

target_words, word_no, word_spellings, word_forms = load_target_words()
print(f"Loaded {len(target_words)} target words, {sum(map(len, word_forms.values())):,} forms")
print(target_words)
assert len(target_words) == 67, f"Expected 67 words, got {len(target_words)}"

# never change the sense, so a token = form + up to 2 clitics. -උත් is written as the vowel sign ුත්.
CLITICS = ["ත්", "ුත්", "ද", "ම", "යි", "කි", "වත්", "ය"]
ENDINGS = [""] + CLITICS + ["".join(p) for p in itertools.product(CLITICS, repeat=2)]

JOINERS = re.compile(r"[\u200C\u200D]")
def match_key(tok):
    """Key used ONLY for matching: NFC + ZWJ/ZWNJ removed, so ප්\u200Dර and ප්ර match the same."""
    return JOINERS.sub("", unicodedata.normalize("NFC", tok))

# WITH word forms: token key -> words / base form
KEY2WORDS = defaultdict(set)
KEY2FORM  = {}
for w in target_words:
    for f in word_forms[w]:
        for e in ENDINGS:
            k = match_key(f + e)
            KEY2WORDS[k].add(w)
            KEY2FORM.setdefault((w, k), f)

# WITHOUT word forms (target word only): just the word's own spelling(s), exact whole token
BASE_KEY2WORD = {match_key(s): w for w in target_words for s in word_spellings[w]}

# same matchers in the shape the extraction helpers (cell 3) use
MATCHERS = {"forms": defaultdict(dict), "base": defaultdict(dict)}
for (w, k), f in KEY2FORM.items():
    MATCHERS["forms"][k][w] = f
for w in target_words:
    for s in word_spellings[w]:
        MATCHERS["base"][match_key(s)][w] = s

TARGETS = target_words
shared = {k: sorted(v) for k, v in KEY2WORDS.items() if len(v) > 1}
print(f"{len(KEY2WORDS):,} token keys | {len(shared)} shared by 2+ words | {len(BASE_KEY2WORD)} target-word spellings")

Loaded 67 target words, 1,370 forms
['දිය', 'දද', 'නා', 'ගත', 'පස / පස්', 'මුව', 'වත', 'අඹ', 'තෙර', 'දන', 'ලිය', 'වණ', 'උර', 'තල', 'නල', 'පෙළ', 'මද', 'මී', 'රැස', 'වර', 'වැට', 'අත', 'ගල', 'උත්තර', 'කඩ', 'කම්', 'කලාපය', 'සුර', 'බල', 'නද', 'සත්', 'සල් / සල', 'පත්', 'සක', 'පැහ, පැහැ', 'තුරු', 'තිර', 'සඳ', 'සිළු', 'නුඹ', 'හස', 'නිදන', 'පණ', 'පිය', 'දත්', 'කලි', 'පට', 'රද', 'දෙස්', 'සමර', 'වස්', 'සර', 'කුළු', 'සිරි', 'කර', 'කමල් / කමල', 'කන', 'රජ', 'රස', 'රා', 'වැඩ', 'දල', 'ලේ', 'සී', 'වන', 'ලද', 'කල/කල්']
99,724 token keys | 0 shared by 2+ words | 72 target-word spellings


3. 3. cleaning, sentence splitting, scanner with checkpoints

In [7]:
import random, hashlib, pickle, time
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

URL_PATTERN      = re.compile(r"https?://\S+|www\.\S+")
SINHALA_CHAR     = re.compile(r"[\u0D80-\u0DFF]")
NOISE_CHARS      = re.compile(r"[\u00AD\u200B\uFEFF]")
LINE_SPLIT       = re.compile(r"[\r\n]+")
SENTENCE_SPLIT   = re.compile(r"(?<=[.!?\u0964])\s*")
METADATA_PATTERN = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}"
    r"|\d{1,2}:\d{2}\s*(AM|PM)", re.IGNORECASE)
TOKEN_PATTERN    = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
MIN_CHARS, MAX_CHARS, MIN_SINHALA_RATIO = 15, 400, 0.5

def segment(text):
    """Document -> clean sentences (lines first, then . ! ? \u0964)."""
    text = URL_PATTERN.sub("", NOISE_CHARS.sub("", unicodedata.normalize("NFC", text or "")))
    out = []
    for line in LINE_SPLIT.split(text):
        for s in SENTENCE_SPLIT.split(re.sub(r"\s+", " ", line).strip()):
            s = s.strip()
            if MIN_CHARS <= len(s) <= MAX_CHARS:
                n = len(SINHALA_CHAR.findall(s))
                if n / len(s) >= MIN_SINHALA_RATIO and not METADATA_PATTERN.search(s):
                    out.append(s)
    return out

def dedup_key(s):
    return re.sub(r"[^\u0D80-\u0DFF0-9]", "", match_key(s))

class Scanner:
    """For ONE matcher: keeps a random sample (reservoir, Algorithm R) of up to RESERVOIR_K sentences
    per word, and counts ALL distinct matching sentences per word / form / sub-corpus."""
    def __init__(self, name, mode, k=RESERVOIR_K, seed=RANDOM_SEED):
        self.name, self.mode, self.k, self.rng = name, mode, k, random.Random(seed)
        self.pool, self.seen = defaultdict(list), defaultdict(set)
        self.n, self.forms, self.sub_n = Counter(), Counter(), Counter()

    def add(self, sent, h, sub_source, doc_id):
        for tok in TOKEN_PATTERN.findall(sent):
            for word, form in MATCHERS[self.mode].get(match_key(tok), {}).items():
                if h in self.seen[word]:
                    continue
                self.seen[word].add(h)
                self.n[word] += 1; self.forms[(word, form)] += 1; self.sub_n[(word, sub_source)] += 1
                row = {"No": word_no[word], "word": word, "matched": tok, "form": form, "source": self.name,
                       "sub_source": sub_source, "doc_id": doc_id, "sentence": sent}
                pool = self.pool[word]
                if len(pool) < self.k:
                    pool.append(row)
                else:
                    j = self.rng.randint(1, self.n[word])
                    if j <= self.k:
                        pool[j - 1] = row

def extract(corpus, modes, dataset, text_key="text", sub_key=None, id_key=None, max_docs=None, every=100_000):
    """Scan one corpus ONCE and fill one Scanner per mode ("forms" and/or "base").
    Saves a checkpoint every `every` docs; re-running the cell resumes after a Colab disconnect."""
    ckpt = f"{OUT}/ckpt_{corpus}_{'_'.join(modes)}.pkl"
    def save():
        pickle.dump({**state, "scanners": {m: sc.__dict__ for m, sc in state["scanners"].items()}}, open(ckpt, "wb"))
    if os.path.exists(ckpt):
        state = pickle.load(open(ckpt, "rb"))
        for m, d in state["scanners"].items():
            sc = Scanner.__new__(Scanner); sc.__dict__.update(d); state["scanners"][m] = sc
        print(f"Resuming {corpus} after {state['docs_done']:,} documents")
        dataset = dataset.skip(state["docs_done"])
    else:
        state = {"docs_done": 0, "docs": Counter(), "scanners": {m: Scanner(corpus, m) for m in modes}}
    t0 = time.time()
    for rec in tqdm(dataset, desc=f"{corpus} [{', '.join(modes)}]", initial=state["docs_done"]):
        if max_docs and state["docs_done"] >= max_docs:
            break
        sub = (rec.get(sub_key) or corpus) if sub_key else corpus
        doc_id = rec.get(id_key) if id_key else state["docs_done"]
        state["docs_done"] += 1; state["docs"][sub] += 1
        for sent in segment(rec.get(text_key)):
            h = int.from_bytes(hashlib.blake2b(dedup_key(sent).encode(), digest_size=8).digest(), "big")
            for sc in state["scanners"].values():
                sc.add(sent, h, sub, doc_id)
        if state["docs_done"] % every == 0:
            save()
    save()
    print(f"{corpus}: {state['docs_done']:,} docs in {(time.time()-t0)/60:.1f} min | docs per sub-corpus: {dict(state['docs'])}")
    for m, sc in state["scanners"].items():
        save_outputs(sc, state)
    return state["scanners"]

def save_outputs(sc, state):
    """Per corpus + mode: sampled sentences (csv + annotation-tool txt) and a summary workbook."""
    tag = f"{sc.name}_{'with_forms' if sc.mode == 'forms' else 'target_word_only'}"
    rows = [r for w in TARGETS for r in sc.pool[w]]
    df = pd.DataFrame(rows, columns=["No", "word", "matched", "form", "source", "sub_source", "doc_id", "sentence"])
    df.to_csv(f"{OUT}/{tag}_sentences.csv", index=False, encoding="utf-8")
    with open(f"{OUT}/{tag}_sentences.txt", "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"s_word: {w}\nsentences: "
                            + "; ".join(r["sentence"].replace(";", ",") for r in sc.pool[w]) for w in TARGETS))
    summary = pd.DataFrame([{"No": word_no[w], "word": w, "matches_in_corpus": sc.n[w],
                             "sampled": len(sc.pool[w]),
                             "by_form": ", ".join(f"{f} ({c})" for (ww, f), c in sc.forms.most_common() if ww == w),
                             "by_sub_corpus": ", ".join(f"{s} ({c})" for (ww, s), c in sc.sub_n.most_common() if ww == w)}
                            for w in TARGETS])
    with pd.ExcelWriter(f"{OUT}/{tag}_summary.xlsx") as xw:
        summary.to_excel(xw, sheet_name="per_word", index=False)
        pd.DataFrame([{"sub_corpus": k, "documents": v} for k, v in state["docs"].items()]
                     ).to_excel(xw, sheet_name="documents", index=False)
    zero = list(summary.loc[summary.matches_in_corpus == 0, "word"])
    print(f"  [{tag}] {len(df):,} sentences saved | {summary.matches_in_corpus.sum():,} matches in corpus"
          f" | words with 0: {zero}")